In [3]:
import pandas as pd
import re 
from collections import Counter 

In [4]:
df = pd.read_csv('messages_clean_ALL.csv')

In [5]:
avg_words = df['cleaned_Message'].dropna().str.split().str.len().mean()
avg_words

np.float64(55.48122529644269)

In [6]:
texts = df['cleaned_Message'].dropna().astype(str).tolist()

#dropna() to remove any rows with missing values in the "Full Text" column, and astype(str) to ensure that all entries are treated as strings. 
#Finally, we convert the resulting Series to a list using tolist() to get a list of all the transcripts.

print(f"Number of transcripts: {len(texts)}")
df.head()


Number of transcripts: 2024


,Channel Name,Date,Time,Link,Message,cleaned_Message,tokenized_Message,lemmatized_Message
0,Collectif Némésis,2026-03-27,19:17:40,https://t.me/NemesisParis/2722,"L’homme, de type maghrébin, l’aurait suivie da...",l homme de type maghrebin l aurait suivie dans...,"['l', 'homme', 'de', 'type', 'maghrebin', 'l',...","['homme', 'type', 'maghrebin', 'suivie', 'rue'..."
1,Collectif Némésis,2026-03-26,19:57:46,https://t.me/NemesisParis/2721,On continue le tour de nos belles sections ave...,on continue le tour de nos belles sections ave...,"['on', 'continue', 'le', 'tour', 'de', 'nos', ...","['continue', 'tour', 'no', 'belle', 'section',..."
2,Collectif Némésis,2026-03-26,16:32:29,https://t.me/NemesisParis/2720,Le CIO rétabli les tests génétiques de féminit...,le cio retabli les tests genetiques de feminit...,"['le', 'cio', 'retabli', 'les', 'tests', 'gene...","['cio', 'retabli', 'test', 'genetiques', 'femi..."
3,Collectif Némésis,2026-03-24,18:21:19,https://t.me/NemesisParis/2719,Notre section Bordeaux 🪽❤️,notre section bordeaux,"['notre', 'section', 'bordeaux']","['section', 'bordeaux']"
4,Collectif Némésis,2026-03-23,16:48:48,https://t.me/NemesisParis/2718,"Trois hommes, intervenus pour défendre des fem...",trois hommes intervenus pour defendre des femm...,"['trois', 'hommes', 'intervenus', 'pour', 'def...","['trois', 'hommes', 'intervenus', 'defendre', ..."


In [7]:
#texts = df['cleaned_Message'].to_list()  # Convert the "Full Text" column to a list of strings
texts= df['lemmatized_Message']

In [21]:
target_phrases = ["violence"]
window_size = 10 #how many words before and after the target phrase to include in the context window

In [22]:
co_terms = Counter()
gbv_counts = Counter()
examples = []  
matches = 0

window_size = 10  

GBV_FRENCH_LEXICON = {
    # Core violence
    "violer",
    "sodomie",
    "pénétration",
    "fellation",
    "agression",
    "abus",
    "meurtre",
    "meurtri",
    "tuer",
    "assassinat",
    "barbarie",
    "torture",
    "supplice",
    # Physical acts
    "coup",
    "frapper",
    "tabasser",
    "lyncher",
    "rouer",
    "poing",
    "claque",
    "gifler",
    "étrangler",
    "strangulation",
    "étouffer",
    "ligoter",
    "attacher",
    "suspendre",
    "mordre",
    "mordue",
    "tirer",
    "pousser",
    "exhibition",
    # Physical state & weapons
    "hématome",
    "contusion",
    "dermabrasion",
    "tuméfié",
    "gonflé",
    "déformé",
    "blessé",
    "ensanglanté",
    "sang",
    "couteau",
    "lame",
    "arme",
    "calibre",
    "corps",
    "nu",
    "nue",
    "déshabiller",
    "incapacité",
    # Predatory & Exploitation
    "proie",
    "repérer",
    "suivre",
    "traquer",
    "menace",
    "punition",
    "bourreau",
    "filmer",
    "publier",
    "darkweb",
    "film",
    "vidéo",
    "bande",
    "criminel",
    "faveur",
    # Slurs & Dehumanization
    "pute",
    "salope",
    "chienne",
    "rut",
    "baise",
    "niquer",
    "insulte",
    "consommer",
    "animal",
    "sacrifier",
    "vice",
    # Emotional/Qualitative
    "horreur",
    "choquant",
    "choc",
    "effroyable",
    "humilié",
    "humiliation",
    "drame",
    "insoutenable",
    "sordide",
    "abominable",
    "brutalement",
    "hurler",
    "hurlant",
    "irréparable",
    "sacrifié",
}

target_words = set()
for phrase in target_phrases:
    target_words.update(phrase.lower().split())

for text in texts:
    words = (
        re.findall(r"\b\w+\b", text.lower())
        if isinstance(text, str)
        else [str(w).lower() for w in text]
    )

    for i, word in enumerate(words):
        single_word_match = word in target_phrases
        bigram_match = (i < len(words) - 1) and (
            f"{word} {words[i+1]}" in target_phrases
        )

        if single_word_match or bigram_match:
            matches += 1
            phrase_length = 2 if bigram_match else 1
            matched_phrase = (
                f"{word} {words[i+1]}" if bigram_match else word
            )

            # window for examples
            if len(examples) < 5:
                left_words = words[max(0, i - window_size) : i]
                right_words = words[
                    i
                    + phrase_length : min(
                        len(words), i + phrase_length + window_size
                    )
                ]

                left_str = " ".join(left_words)
                right_str = " ".join(right_words)

                examples.append(
                    f"... {left_str} [{matched_phrase.upper()}] {right_str} ..."
                )

            # context window
            start = max(0, i - window_size)
            end = min(len(words), i + phrase_length + window_size)
            context_words = words[start:end]

            for w in context_words:
                if (
                    w.isalpha()
                    and w not in target_words
                    and len(w) > 2
                ):
                    co_terms[w] += 1

                    if w in GBV_FRENCH_LEXICON:
                        gbv_counts[w] += 1

# results
print(f"Total target matches found: {matches}\n")

print("--- top 10 overall context terms ---")
print(co_terms.most_common(10))

print("\n--- top GBV lexicon words appearing near target ---")
print(gbv_counts.most_common)

# examples section
print("\n--- sample context examples ---")
if examples:
    for idx, ex in enumerate(examples, 1):
        print(f"{idx}. {ex}")
else:
    print("No examples captured (0 matches found).")

Total target matches found: 284

--- top 10 overall context terms ---
[('ete', 74), ('conjugales', 72), ('femmes', 63), ('sexuelles', 56), ('viol', 47), ('condamne', 42), ('deja', 42), ('contre', 33), ('etait', 31), ('homme', 30)]

--- top GBV lexicon words appearing near target ---
<bound method Counter.most_common of Counter({'coup': 18, 'menace': 16, 'agression': 8, 'meurtre': 6, 'arme': 5, 'couteau': 5, 'torture': 3, 'barbarie': 3, 'poing': 3, 'tuer': 3, 'exhibition': 2, 'violer': 2, 'claque': 1, 'punition': 1, 'strangulation': 1, 'sang': 1, 'suivre': 1, 'insoutenable': 1, 'fellation': 1})>

--- sample context examples ---
1. ... dont raphael arnault fondateur meme raphael arnault a ete condamne [VIOLENCE] volontaires reunion plus evident arnault lfi portent immense responsabilite quant ...
2. ... comme constate nemesis fait face a offensive [VIOLENCE] inedite apres meurtre barbare quentin lynche a mort assistant parlementaires ...
3. ... ose denoncer leurs agresseurs repression ve